In [4]:
import torch
import torch.nn.functional as F
from rich import print as print
from torch.nn import Module
from torchvision import transforms
from torchvision.datasets import CIFAR10

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0

In [5]:
image_transforms = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(
            (0.5,0.5,0.5),
            (0.5,0.5,0.5)
        )
    ]
)

print(image_transforms)

Compose(
    ToTensor()
    Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
)

In [6]:
trainset = CIFAR10(
    root="./",
    train=True,
    transform=image_transforms,
    download=True
)

testset = CIFAR10(
    root="./",
    train=False,
    transform=image_transforms,
    download=True
)

100%|██████████| 170M/170M [30:27<00:00, 93.3kB/s]   


In [7]:
print(trainset)
print(testset)

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: ./
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [8]:
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=100,
    shuffle=True,
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=100,
    shuffle=False,
)

In [9]:
trainset.data[0].shape

(32, 32, 3)

### Model code

In [10]:
from torch.nn import Conv2d, Linear, MaxPool2d


class Net(Module):
    def __init__(self):
        super().__init__()

        self.conv1 = Conv2d(in_channels=3, out_channels= 6, kernel_size=5)
        self.pool = MaxPool2d(kernel_size=2, stride= 2)
        self.conv2 = Conv2d(in_channels=6, out_channels= 16, kernel_size= 5)
        self.fc1 = Linear(in_features=16*5*5,out_features= 120)
        self.fc2 = Linear(in_features=120, out_features= 84)
        self.fc3 = Linear(in_features= 84, out_features=10)

    def forward(self,x ):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))


        x = x.view(-1, 16*5*5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [11]:
model1 = Net().to(device)
print(model1)
print(model1.parameters)

Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

<bound method Module.parameters of Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)>

In [12]:
total_params = sum(p.numel() for p in model1.parameters())
print(total_params)  

62006

In [13]:
from torch import nn, optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    params=model1.parameters(),
    lr = 0.01, 
    momentum=0.9
)

In [14]:
print(criterion)
print(optimizer)

CrossEntropyLoss()

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0
)

In [15]:
dataiter = iter(trainloader)
images, labels = next(dataiter)
images, labels = images.to(device), labels.to(device)
outputs = model1(images)
loss = criterion(outputs, labels)
print("Initial loss:", loss.item())

Initial loss: 2.3139448165893555

In [16]:
for epoch in range(2):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model1(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1} loss: {running_loss / len(trainloader):.4f}")

print("Final batch loss after training:", loss.item())

Epoch 1 loss: 1.9924

Epoch 2 loss: 1.5053

Final batch loss after training: 1.4021823406219482

In [17]:
correct = 0
total = 0

with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = model1(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print("Accuracy:", accuracy)

Accuracy: 0.4988